In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from keras.models import load_model

In [2]:
df = pd.read_csv("IMDb_TOP50_Reviews.csv")
df.head()

,Movie,Review Text,IMDb Rating
0,The Shawshank Redemption,A profound masterpiece about the resilience of...,10
1,The Godfather,The pinnacle of crime drama. Coppola's directi...,10
2,The Dark Knight,Nolan redefined the superhero genre. Ledger’s ...,9
3,Breaking Bad,The greatest character study ever televised. W...,10
4,The Sopranos,Revolutionized television. James Gandolfini’s ...,10


In [3]:
import re
import nltk
from nltk.corpus import stopwords
stopwords_list = set(stopwords.words('english'))


TAG_RE = re.compile(r'<[^>]+>')

def remove_tags(text):
    return TAG_RE.sub('', text)
    

class CustomPreprocess():

    def __init__(self):
        pass

    def preprocess_text(self,sen):
        sen = sen.lower()
        
        # Remove html tags
        sentence = remove_tags(sen)

        # Remove punctuations and numbers
        sentence = re.sub('[^a-zA-Z]', ' ', sentence)
        
        # Single character removal
        sentence = re.sub(r"\s+[a-zA-Z]\s+", ' ', sentence)

        # Remove multiple spaces
        sentence = re.sub(r'\s+', ' ', sentence)
        
        # Remove Stopwords
        pattern = re.compile(r'\b(' + r'|'.join(stopwords_list) + r')\b\s*')
        sentence = pattern.sub('', sentence)
        
        return sentence

In [4]:
custom = CustomPreprocess()
unseen_reviews = df['Review Text']

unseen_processed = []
for review in unseen_reviews:
    review = custom.preprocess_text(review)
    unseen_processed.append(review)

In [5]:
unseen_processed[:3]

['profound masterpiece resilience human spirit chemistry robbins freeman legendary ending perhaps satisfying cinematic history ',
 'pinnacle crime drama coppola direction brando performance create haunting beautiful portrait power family never surpassed ',
 'nolan redefined superhero genre ledger joker chaotic force nature makes high stakes crime thriller comic book movie ']

In [6]:
# Loading
import io
import json
from tensorflow.keras.preprocessing.text import tokenizer_from_json, Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import load_model

with open('tokenizer.json') as f:
    data = json.load(f)
    loaded_tokenizer = tokenizer_from_json(data)

In [7]:
from tensorflow.keras.preprocessing.text import Tokenizer
# tokenizer = Tokenizer()
# tokenizer.fit_on_texts(unseen_processed)
# unseen_tokenized  = tokenizer.texts_to_sequences(unseen_processed)
unseen_tokenized = loaded_tokenizer.texts_to_sequences(unseen_processed)


In [8]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
unseen_padded = pad_sequences(unseen_tokenized, padding='post', maxlen=100)


In [9]:
unseen_padded[:2]

array([[ 3123,   770, 27210,   271,   986,  1080,  4878,  3023,  2452,
          154,   262,  2209,  1157,   352,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,
            0],
       [11273,   640,   323,  5820,   331,  3491,   129,   848,  2204,
          197,  3063,   511,   118,    34,  9246,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     

In [10]:
model = load_model("sentiment_classifier2.h5")

In [11]:
try:
    pred = model.predict(unseen_padded, batch_size=len(unseen_padded))
except ValueError:
    # If that fails, the model might only want 1 review at a time
    print("🔄 Standard prediction failed, trying manual loop...")
    pred = []
    for review in unseen_padded:
        # Reshape to (1, 100) because the model expects [Batch_Size, Max_Length]
        single_pred = model.predict(review.reshape(1, 100), verbose=0)
        pred.append(single_pred[0])
    pred = np.array(pred)

print("✅ Prediction complete!")
print(pred[:5]) # Show first 5 results

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 475ms/step
✅ Prediction complete!
[[0.98957974]
 [0.9883966 ]
 [0.9839054 ]
 [0.9892361 ]
 [0.9885676 ]]


In [17]:
pred_df = df.drop(columns=["Unnamed: 0"], errors="ignore")
pred_df

,Movie,Review Text,IMDb Rating
0,The Shawshank Redemption,A profound masterpiece about the resilience of...,10
1,The Godfather,The pinnacle of crime drama. Coppola's directi...,10
2,The Dark Knight,Nolan redefined the superhero genre. Ledger’s ...,9
3,Breaking Bad,The greatest character study ever televised. W...,10
4,The Sopranos,Revolutionized television. James Gandolfini’s ...,10
5,Band of Brothers,"A visceral, deeply moving tribute to the men w...",10
6,Chernobyl,"A chilling, claustrophobic look at human error...",9
7,Avatar: The Last Airbender,Perfect storytelling for all ages. It balances...,10
8,The Wire,"A gritty, uncompromising look at the American ...",9
9,Game of Thrones,"Despite a divisive ending, the first six seaso...",8


In [18]:
pred_df["Predicted Sentiment"] = np.round(pred*10,1)
pred_df

,Movie,Review Text,IMDb Rating,Predicted Sentiment
0,The Shawshank Redemption,A profound masterpiece about the resilience of...,10,9.9
1,The Godfather,The pinnacle of crime drama. Coppola's directi...,10,9.9
2,The Dark Knight,Nolan redefined the superhero genre. Ledger’s ...,9,9.8
3,Breaking Bad,The greatest character study ever televised. W...,10,9.9
4,The Sopranos,Revolutionized television. James Gandolfini’s ...,10,9.9
5,Band of Brothers,"A visceral, deeply moving tribute to the men w...",10,9.9
6,Chernobyl,"A chilling, claustrophobic look at human error...",9,9.9
7,Avatar: The Last Airbender,Perfect storytelling for all ages. It balances...,10,9.9
8,The Wire,"A gritty, uncompromising look at the American ...",9,9.9
9,Game of Thrones,"Despite a divisive ending, the first six seaso...",8,9.9


In [15]:
pred_label = []
for i in list(pred_df["Predicted Sentiment"]):
    if i <= 5:
        pred_label.append("Negative")
    
    else:
        pred_label.append("Positive")

In [16]:
pred_df["Predicted Review Sentiment"] = pred_label
pred_df

,Movie,Review Text,IMDb Rating,Predicted Sentiment,Predicted Review Sentiment
0,The Shawshank Redemption,A profound masterpiece about the resilience of...,10,9.9,Positive
1,The Godfather,The pinnacle of crime drama. Coppola's directi...,10,9.9,Positive
2,The Dark Knight,Nolan redefined the superhero genre. Ledger’s ...,9,9.8,Positive
3,Breaking Bad,The greatest character study ever televised. W...,10,9.9,Positive
4,The Sopranos,Revolutionized television. James Gandolfini’s ...,10,9.9,Positive
5,Band of Brothers,"A visceral, deeply moving tribute to the men w...",10,9.9,Positive
6,Chernobyl,"A chilling, claustrophobic look at human error...",9,9.9,Positive
7,Avatar: The Last Airbender,Perfect storytelling for all ages. It balances...,10,9.9,Positive
8,The Wire,"A gritty, uncompromising look at the American ...",9,9.9,Positive
9,Game of Thrones,"Despite a divisive ending, the first six seaso...",8,9.9,Positive
